# Discursos

In [ ]:
import requests
import time

print(f"🚀 Iniciando carga estável para {len(ids_deputados)} deputados...")
inicio_execucao = time.time()
total_geral = 0

for idx, id_dep in enumerate(ids_deputados, 1):
    pagina = 1
    discursos_dep = []

    # Coleta de todas as páginas de um único deputado
    while True:
        url_api = f"https://dadosabertos.camara.leg.br/api/v2/deputados/{id_dep}/discursos"
        params = {'dataInicio': '2024-01-01', 'itens': 100, 'pagina': pagina}

        try:
            response = requests.get(url_api, params=params, timeout=20)
            if response.status_code == 200:
                dados = response.json().get('dados', [])
                if not dados: break

                for d in dados:
                    discursos_dep.append((
                        id_dep, d.get('transcricao'), d.get('tipoDiscurso'),
                        d['faseEvento'].get('titulo') if d.get('faseEvento') else None,
                        d.get('dataHoraInicio'), d.get('urlTexto'), d.get('keywords')
                    ))

                if not any(l['rel'] == 'next' for l in response.json().get('links', [])): break
                pagina += 1
            else: break
        except Exception as e:
            print(f"⚠️ Erro de rede no ID {id_dep}: {e}")
            break

    # Gravação imediata no banco para manter a conexão ativa
    if discursos_dep:
        try:
            cursor.executemany(sql_insert, discursos_dep)
            conn.commit()
            total_geral += len(discursos_dep)

            # Feedback visual para monitoramento
            if idx % 10 == 0:
                print(f"✅ {idx}/{len(ids_deputados)} processados. Total acumulado: {total_geral}")
        except Exception as e:
            print(f"❌ Erro crítico no ID {id_dep}: {e}")
            conn.rollback()
            break

tempo_final = (time.time() - inicio_execucao) / 60
print(f"\n🏁 Carga finalizada! Total: {total_geral} discursos em {tempo_final:.2f} min.")